In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numba import njit
import seaborn as sns
import bayesflow as bf

import sys
sys.path.append("../")

from src.generative_models import sample_pt_model, sample_mvl_model, get_choice, get_pt_utility, get_mvl_utility
from src.context import get_context
from src.priors import sample_pt_prior, sample_mvl_prior

In [49]:
which = 0
gamble_df = pd.read_csv(f"../data/three_outcome_lotteries_{which}.csv")

In [ ]:
# context = gamble_df.loc[gamble_df.ev_diff == 0][[
context = gamble_df[[
    'outcome_a1', 'outcome_a2', 'outcome_a3',
    'outcome_b1', 'outcome_b2', 'outcome_b3'
]].to_numpy()
context.shape

In [198]:
NUM_SIM = 200
NUM_SAMPLES = 2000

In [199]:
pt_la_agent_data = np.zeros((NUM_SIM, context.shape[0], 7))
mv_la_agent_data = np.zeros((NUM_SIM, context.shape[0], 7))

pt_var_agent_data = np.zeros((NUM_SIM, context.shape[0], 7))
mv_var_agent_data = np.zeros((NUM_SIM, context.shape[0], 7))

pt_both_agent_data = np.zeros((NUM_SIM, context.shape[0], 7))
mv_both_agent_data = np.zeros((NUM_SIM, context.shape[0], 7))

In [200]:
tau = 0.8
beta = 0.4
theta_pt_la = np.array([2, 1, tau])
theta_mv_la = np.array([0, beta, tau])

theta_pt_var = np.array([1, 0.4, tau])
theta_mv_var = np.array([beta, 0, tau])

theta_pt_both = np.array([2, 0.4, tau])
theta_mv_both = np.array([beta, beta, tau])

In [201]:
context_trans = context / 200
for i in range(NUM_SIM):
    pt_la_agent_data[i, :, 0] = sample_pt_model(theta_pt_la, context)
    pt_la_agent_data[i, :, 1:] = context_trans
    mv_la_agent_data[i, :, 0] = sample_mvl_model(theta_mv_la, context)
    mv_la_agent_data[i, :, 1:] = context_trans

    pt_var_agent_data[i, :, 0] = sample_pt_model(theta_pt_var, context)
    pt_var_agent_data[i, :, 1:] = context_trans
    mv_var_agent_data[i, :, 0] = sample_mvl_model(theta_mv_var, context)
    mv_var_agent_data[i, :, 1:] = context_trans

    pt_both_agent_data[i, :, 0] = sample_pt_model(theta_pt_both, context)
    pt_both_agent_data[i, :, 1:] = context_trans
    mv_both_agent_data[i, :, 0] = sample_mvl_model(theta_mv_both, context)
    mv_both_agent_data[i, :, 1:] = context_trans

In [202]:
LABELS = [
    "PT loss averse", "PT var averse", "PT both aversions",
    "MV loss averse", "MV var averse", "MV both aversions",
]

In [203]:
means = np.array([
    pt_la_agent_data[: , :, 0].mean(axis=1).mean(), pt_var_agent_data[: , :, 0].mean(axis=1).mean(), pt_both_agent_data[: , :, 0].mean(axis=1).mean(),
    mv_la_agent_data[: , :, 0].mean(axis=1).mean(), mv_var_agent_data[: , :, 0].mean(axis=1).mean(), mv_both_agent_data[: , :, 0].mean(axis=1).mean()

])

stds = np.array([
    pt_la_agent_data[: , :, 0].mean(axis=1).std(), pt_var_agent_data[: , :, 0].mean(axis=1).std(), pt_both_agent_data[: , :, 0].mean(axis=1).std(),
    mv_la_agent_data[: , :, 0].mean(axis=1).std(), mv_var_agent_data[: , :, 0].mean(axis=1).std(), mv_both_agent_data[: , :, 0].mean(axis=1).std()

])

In [204]:
FONTSIZE_1 = 24
FONTSIZE_2 = 18
FONTSIZE_3 = 16

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(LABELS, means, yerr=stds, fmt='o', capsize=5, color='maroon', ecolor='maroon')
ax.axhline(y=0.5, linestyle='--', color='gray', linewidth=1.5)
ax.set_xlabel("Agent Types", fontsize=FONTSIZE_1, labelpad=10)
ax.set_ylabel("P(higher var option)", fontsize=FONTSIZE_1, labelpad=10)
ax.grid(True, linestyle='--', alpha=0.6)
ax.tick_params(axis='both', which='major', labelsize=FONTSIZE_2) 
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
sns.despine()
# plt.savefig("../plots/agent_simulation.pdf", dpi=300, bbox_inches="tight")

In [206]:
context_gen = bf.simulation.ContextGenerator(
    batchable_context_fun=get_context
)

## PT Model

In [207]:
param_names = [r'$\lambda$', r'$\alpha$', r'$\tau$']

PT_PRIOR_MEANS = np.array([1.7, 0.7, 0.5])
PT_PRIOR_STDS = np.array([0.7, 0.3, 0.5])

In [208]:
prior = bf.simulation.Prior(
    batch_prior_fun=sample_pt_prior,
    param_names=param_names,
)

In [ ]:
simulator = bf.simulation.Simulator(
    simulator_fun=sample_pt_model,
    context_generator=context_gen
)

model = bf.simulation.GenerativeModel(
    prior=prior,
    simulator=simulator,
    name="pt_model"
)

In [210]:
summary_net = bf.networks.SetTransformer(input_dim=7, summary_dim=32)

inference_net = bf.networks.InvertibleNetwork(
    num_params=len(prior.param_names),
    coupling_settings={"dense_args": dict(kernel_regularizer=None), "dropout": False},
)

In [211]:
def configurator(forward_dict):
    out_dict = {}
    data = forward_dict["sim_data"][:, :, None]
    context = np.array(forward_dict["sim_batchable_context"]) / 200
    out_dict["summary_conditions"] = np.c_[data, context].astype(np.float32)
    params = forward_dict["prior_draws"].astype(np.float32)
    out_dict["parameters"] = (params - PT_PRIOR_MEANS) / PT_PRIOR_STDS
    return out_dict

In [ ]:
amortizer = bf.amortizers.AmortizedPosterior(inference_net, summary_net)

trainer = bf.trainers.Trainer(
    generative_model=model, 
    amortizer=amortizer, 
    configurator=configurator, 
    checkpoint_path=f"../checkpoints/{model.name}",
    max_to_keep=1
)

In [213]:
pt_la_agent_post_z = amortizer.sample({"summary_conditions": pt_la_agent_data.astype(np.float32)}, NUM_SAMPLES)
pt_var_agent_post_z = amortizer.sample({"summary_conditions": pt_var_agent_data.astype(np.float32)}, NUM_SAMPLES)
pt_both_agent_post_z = amortizer.sample({"summary_conditions": pt_both_agent_data.astype(np.float32)}, NUM_SAMPLES)

In [214]:
pt_la_agent_post = pt_la_agent_post_z * PT_PRIOR_STDS + PT_PRIOR_MEANS
pt_var_agent_post = pt_var_agent_post_z * PT_PRIOR_STDS + PT_PRIOR_MEANS
pt_both_agent_post = pt_both_agent_post_z * PT_PRIOR_STDS + PT_PRIOR_MEANS

In [ ]:
fig = plt.figure(figsize=(12, 3))
axs = fig.subplots(nrows=1, ncols=3)
for i, ax in enumerate(axs):
    sns.histplot(pt_la_agent_post.mean(axis=1)[:, i], ax=ax, color='maroon', kde=True, bins=20)
    ax.axvline(x=theta_pt_la[i], linestyle='--', color='black', linewidth=2.5)
    ax.set_title(param_names[i], fontsize=FONTSIZE_1)
    ax.tick_params(axis='both', which='major', labelsize=FONTSIZE_3)
    if i == 0:
        ax.set_ylabel("Density", fontsize=FONTSIZE_2, labelpad=10)
    else:
        ax.set_ylabel("")
sns.despine()
plt.tight_layout()

In [ ]:
fig = plt.figure(figsize=(12, 3))
axs = fig.subplots(nrows=1, ncols=3)
for i, ax in enumerate(axs):
    sns.histplot(pt_var_agent_post.mean(axis=1)[:, i], ax=ax, color='maroon', kde=True, bins=20)
    ax.axvline(x=theta_pt_var[i], linestyle='--', color='black', linewidth=2.5)
    ax.set_title(param_names[i], fontsize=FONTSIZE_1)
    ax.tick_params(axis='both', which='major', labelsize=FONTSIZE_3)
    if i == 0:
        ax.set_ylabel("Density", fontsize=FONTSIZE_2, labelpad=10)
    else:
        ax.set_ylabel("")
sns.despine()
plt.tight_layout()

In [ ]:
fig = plt.figure(figsize=(12, 3))
axs = fig.subplots(nrows=1, ncols=3)
for i, ax in enumerate(axs):
    sns.histplot(pt_both_agent_post.mean(axis=1)[:, i], ax=ax, color='maroon', kde=True, bins=20)
    ax.axvline(x=theta_pt_both[i], linestyle='--', color='black', linewidth=2.5)
    ax.set_title(param_names[i], fontsize=FONTSIZE_1)
    ax.tick_params(axis='both', which='major', labelsize=FONTSIZE_3)
    if i == 0:
        ax.set_ylabel("Density", fontsize=FONTSIZE_2, labelpad=10)
    else:
        ax.set_ylabel("")
sns.despine()
plt.tight_layout()

## MVL Model

In [232]:
parameter_names = [r'$\b_{var}$', r'$\b_{loss}$', r'$\tau$']

MVL_PRIOR_MEANS = np.array([0.3, 0.3, 0.5])
MVL_PRIOR_STDS = np.array([0.26, 0.26, 0.5])

In [218]:
prior = bf.simulation.Prior(
    batch_prior_fun=sample_mvl_prior,
    param_names=parameter_names
)

In [ ]:
simulator = bf.simulation.Simulator(
    simulator_fun=sample_mvl_model,
    context_generator=context_gen
)

model = bf.simulation.GenerativeModel(
    prior=prior,
    simulator=simulator,
    name="mvl_model"
)

In [220]:
summary_net = bf.networks.SetTransformer(input_dim=7, summary_dim=32)

inference_net = bf.networks.InvertibleNetwork(
    num_params=len(prior.param_names),
    coupling_settings={"dense_args": dict(kernel_regularizer=None), "dropout": False},
)

In [221]:
def configurator(forward_dict):
    out_dict = {}
    data = forward_dict["sim_data"][:, :, None]
    context = np.array(forward_dict["sim_batchable_context"]) / 200
    out_dict["summary_conditions"] = np.c_[data, context].astype(np.float32)
    params = forward_dict["prior_draws"]
    out_dict["parameters"] = ((params - MVL_PRIOR_MEANS) / MVL_PRIOR_STDS).astype(np.float32)
    return out_dict

In [ ]:
amortizer = bf.amortizers.AmortizedPosterior(inference_net, summary_net)

trainer = bf.trainers.Trainer(
    generative_model=model, 
    amortizer=amortizer, 
    configurator=configurator, 
    checkpoint_path=f"../checkpoints/{model.name}",
    max_to_keep=1
)

In [223]:
mv_la_agent_post_z = amortizer.sample({"summary_conditions": mv_la_agent_data.astype(np.float32)}, NUM_SAMPLES)
mv_var_agent_post_z = amortizer.sample({"summary_conditions": mv_var_agent_data.astype(np.float32)}, NUM_SAMPLES)
mv_both_agent_post_z = amortizer.sample({"summary_conditions": mv_both_agent_data.astype(np.float32)}, NUM_SAMPLES)

In [224]:
mv_la_agent_post = mv_la_agent_post_z * MVL_PRIOR_STDS + MVL_PRIOR_MEANS
mv_var_agent_post = mv_var_agent_post_z * MVL_PRIOR_STDS + MVL_PRIOR_MEANS
mv_both_agent_post = mv_both_agent_post_z * MVL_PRIOR_STDS + MVL_PRIOR_MEANS

In [ ]:
fig = plt.figure(figsize=(12, 3))
axs = fig.subplots(nrows=1, ncols=3)
for i, ax in enumerate(axs):
    sns.histplot(mv_la_agent_post.mean(axis=1)[:, i], ax=ax, color='maroon', kde=True, bins=20)
    ax.axvline(x=theta_mv_la[i], linestyle='--', color='black', linewidth=2.5)
    ax.set_title(param_names[i], fontsize=FONTSIZE_1)
    ax.tick_params(axis='both', which='major', labelsize=FONTSIZE_3)
    if i == 0:
        ax.set_ylabel("Density", fontsize=FONTSIZE_2, labelpad=10)
    else:
        ax.set_ylabel("")
sns.despine()
plt.tight_layout()

In [ ]:
fig = plt.figure(figsize=(12, 3))
axs = fig.subplots(nrows=1, ncols=3)
for i, ax in enumerate(axs):
    sns.histplot(mv_var_agent_post.mean(axis=1)[:, i], ax=ax, color='maroon', kde=True, bins=20)
    ax.axvline(x=theta_mv_var[i], linestyle='--', color='black', linewidth=2.5)
    ax.set_title(param_names[i], fontsize=FONTSIZE_1)
    ax.tick_params(axis='both', which='major', labelsize=FONTSIZE_3)
    if i == 0:
        ax.set_ylabel("Density", fontsize=FONTSIZE_2, labelpad=10)
    else:
        ax.set_ylabel("")
sns.despine()
plt.tight_layout()

In [ ]:
fig = plt.figure(figsize=(12, 3))
axs = fig.subplots(nrows=1, ncols=3)
for i, ax in enumerate(axs):
    sns.histplot(mv_both_agent_post.mean(axis=1)[:, i], ax=ax, color='maroon', kde=True, bins=20)
    ax.axvline(x=theta_mv_both[i], linestyle='--', color='black', linewidth=2.5)
    ax.set_title(param_names[i], fontsize=FONTSIZE_1)
    ax.tick_params(axis='both', which='major', labelsize=FONTSIZE_3)
    if i == 0:
        ax.set_ylabel("Density", fontsize=FONTSIZE_2, labelpad=10)
    else:
        ax.set_ylabel("")
sns.despine()
plt.tight_layout()